# CLEIDS-Edge — Notebook 07: Final Results & Figures

Consolidates every real result from Notebooks 03—06 into a single `results/all_paper_numbers.json`
and produces the final cross-cutting figures for thesis Chapter 4/5 — the ones that directly support
the three contributions (project brief SS5): accuracy-vs-size-vs-latency (pre/post compression),
baseline comparison across all 5 datasets, and CLEIDS-Edge vs baselines 1—8 on every metric.

**No GPU, no raw data, no model checkpoints needed here** — unlike every other notebook in this
project. All five inputs (`main_results.json`, `tuned_threshold_results.json`, `baseline_results.json`,
`compression_results.json`, `latency_results.json`) are real results already produced by Notebooks
03—06 and already tracked in git, so a plain `git clone`/`pull` is enough; nothing needs restoring
from Drive. This notebook is pure Python + matplotlib/seaborn.

**Validated locally against the real, complete results before being written up here** — not just a
smoke test: the full consolidation and all four figures were run against this project's actual
`results/*.json` files, and each figure was visually inspected, not just checked for "ran without
error." That inspection caught a real bug: an in-plot legend on the efficiency-frontier figure,
placed at `loc="lower left"`, sat exactly on top of SVM's real data point (the fastest model, with a
mid-range F1 — precisely the "lower left" region), silently hiding a genuine result. Fixed by moving
the legend outside the axes entirely. A chart that runs without a traceback is not the same as a
chart that is correct.

**Two known, real findings surfaced independently by the consolidated heatmap**, both consistent with
prior notebooks' documented results, not new anomalies: IoT-23 is uniformly hard (~0.465 tuned F1)
for every single model regardless of architecture (RF, SVM, CNN, LSTM, hybrid) — a genuine
dataset-level ceiling (project brief SS2c), not a CLEIDS-Edge weakness; and Standalone LSTM's real
TON_IoT training collapse (SS2e) is visible as the one clearly lighter cell in that heatmap column
(F1=0.527 vs. 0.81—0.99 everywhere else).

**Scope notes, carried forward honestly from earlier notebooks**: baselines are binary-task only
(Notebook 04's scope) — CLEIDS-Edge's multiclass results have no baseline-comparison counterpart and
no tuned-threshold entry (Youden's J tuning was binary-only, SS2d), so they are reported as
CLEIDS-Edge-only insight in `all_paper_numbers.json`, not blended into the comparison figures. Peak
memory is not included anywhere — found genuinely infeasible to measure in the Colab sandbox (SS3h);
model size is the memory-footprint proxy used throughout.

**No fabricated numbers** — every figure and every entry in `all_paper_numbers.json` is read directly
from the five real results files, with an explicit key-presence validation pass before anything is
plotted (all 5 datasets x 7 baselines in `baseline_results.json`, all expected keys in
`compression_results.json` and `latency_results.json`) — same discipline as every other notebook.

## 1. Repo setup (clone/pull + auth)

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/NehlTech/CLEIDS-Edge.git"
REPO_DIR = "/content/CLEIDS-Edge"

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception as e:
    print(f"[DEBUG] userdata.get('GITHUB_TOKEN') raised {type(e).__name__}: {e}")
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN not found. Add it as a Colab secret (key icon in the left sidebar) "
        "if running in the real Colab UI, or set os.environ['GITHUB_TOKEN'] manually for "
        "this session if running over a proxied connection."
    )

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

AUTH_REMOTE = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", AUTH_REMOTE], check=True)
subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", "obololastkiller@gmail.com"])
subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", "Bright Adu-Boahene"])

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 2. Google Drive mount (backup only -- no data restore needed this time)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/CLEIDS_Edge"
DRIVE_RESULTS = os.path.join(DRIVE_ROOT, "results")
DRIVE_FIGURES = os.path.join(DRIVE_ROOT, "figures")
for d in (DRIVE_RESULTS, DRIVE_FIGURES):
    os.makedirs(d, exist_ok=True)
print("Drive ready at:", DRIVE_ROOT)


## 3. Setup -- load all real results (no GPU, no raw data needed)

All five files below already exist in the repo (git-tracked, produced by Notebooks 03-06) -- loaded
directly, nothing restored from Drive.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATASETS = ["nsl-kdd", "cicids2017", "unsw-nb15", "ton-iot", "iot-23"]
BASELINE_MODELS = ["random_forest", "svm", "standalone_cnn", "standalone_lstm",
                    "nazir2024", "altaie_hoomod2024", "wang2023_dlbilstm"]
MODEL_DISPLAY_ORDER = ["cleids_edge"] + BASELINE_MODELS
DISPLAY_NAMES = {
    "cleids_edge": "CLEIDS-Edge", "random_forest": "Random Forest", "svm": "SVM",
    "standalone_cnn": "Standalone CNN", "standalone_lstm": "Standalone LSTM",
    "nazir2024": "Nazir 2024", "altaie_hoomod2024": "Altaie-Hoomod 2024",
    "wang2023_dlbilstm": "Wang 2023 DL-BiLSTM",
}

with open("results/main_results.json") as f:
    main_results = json.load(f)
with open("results/tuned_threshold_results.json") as f:
    tuned_results = json.load(f)
with open("results/baseline_results.json") as f:
    baseline_results = json.load(f)
with open("results/compression_results.json") as f:
    compression_results = json.load(f)
with open("results/latency_results.json") as f:
    latency_results = json.load(f)

os.makedirs("figures", exist_ok=True)

# ---- validation: every expected key present before anything is plotted ----
assert set(main_results.keys()) == set(DATASETS)
assert set(tuned_results.keys()) == set(DATASETS)
assert set(baseline_results.keys()) == set(BASELINE_MODELS)
for m in BASELINE_MODELS:
    assert set(baseline_results[m].keys()) == set(DATASETS), m
print("[OK] all 5 datasets x 7 baselines present in baseline_results.json")

for name in DATASETS:
    for task in ["binary", "multiclass"]:
        assert f"cleids_edge_{name}_{task}" in compression_results, f"{name}/{task} missing from compression_results"
        for variant in ["__original", "__quantized"]:
            key = f"cleids_edge_{name}_{task}{variant}"
            assert key in latency_results, f"{key} missing from latency_results"
for m in BASELINE_MODELS:
    for name in DATASETS:
        key = f"{m}_{name}_binary"
        assert key in latency_results, f"{key} missing from latency_results"
print("[OK] all expected keys present in compression_results.json and latency_results.json")


## 4. Consolidate into `results/all_paper_numbers.json`

In [ ]:
def get_f1(model, dataset, tuned=True):
    """Tuned-threshold F1 for any model (CLEIDS-Edge or baseline), binary task."""
    if model == "cleids_edge":
        return tuned_results[dataset]["f1"] if tuned else main_results[dataset]["binary"]["f1"]
    key = "tuned_threshold" if tuned else "default_threshold"
    return baseline_results[model][dataset][key]["f1"]


def get_latency(model, dataset, variant="original"):
    if model == "cleids_edge":
        key = f"cleids_edge_{dataset}_binary__{'quantized' if variant == 'quantized' else 'original'}"
        return latency_results[key]
    return latency_results[f"{model}_{dataset}_binary"]


# ---- spot-check a couple of numbers by hand before trusting the pipeline ----
print("[SPOT CHECK] CLEIDS-Edge nsl-kdd tuned F1:", get_f1("cleids_edge", "nsl-kdd"))
print("[SPOT CHECK] Random Forest nsl-kdd tuned F1:", get_f1("random_forest", "nsl-kdd"))
print("[SPOT CHECK] CLEIDS-Edge nsl-kdd latency (original):", get_latency("cleids_edge", "nsl-kdd", "original")["latency_ms_mean"])
print("[SPOT CHECK] CLEIDS-Edge nsl-kdd latency (quantized):", get_latency("cleids_edge", "nsl-kdd", "quantized")["latency_ms_mean"])

all_paper_numbers = {"cleids_edge": {}, "baselines": {}, "metadata": {}}

for name in DATASETS:
    all_paper_numbers["cleids_edge"][name] = {}
    for task in ["binary", "multiclass"]:
        ckpt_name = f"cleids_edge_{name}_{task}"
        entry = {
            "default_threshold": main_results[name][task],
            "compression": compression_results[ckpt_name],
            "efficiency": {
                "original": latency_results[f"{ckpt_name}__original"],
                "quantized_16x8": latency_results[f"{ckpt_name}__quantized"],
            },
        }
        if task == "binary":
            entry["tuned_threshold"] = tuned_results[name]
        all_paper_numbers["cleids_edge"][name][task] = entry

for m in BASELINE_MODELS:
    all_paper_numbers["baselines"][m] = {}
    for name in DATASETS:
        all_paper_numbers["baselines"][m][name] = {
            "default_threshold": baseline_results[m][name]["default_threshold"],
            "tuned_threshold": baseline_results[m][name]["tuned_threshold"],
            "efficiency": latency_results[f"{m}_{name}_binary"],
        }

all_paper_numbers["metadata"] = {
    "datasets": DATASETS,
    "baseline_models": BASELINE_MODELS,
    "model_display_order": MODEL_DISPLAY_ORDER,
    "notes": (
        "Baselines are binary-task only (Notebook 04 scope). CLEIDS-Edge multiclass results have no "
        "tuned_threshold entry (Youden's J tuning was binary-only, see project brief SS2d) and no "
        "baseline comparison counterpart -- CLEIDS-Edge-only insight. Efficiency numbers are from "
        "Notebook 06 (CPU-only, single-thread, batch=1); peak memory was found infeasible to measure "
        "in the Colab sandbox (project brief SS3h) and is not included -- model size (size_mb) is the "
        "practical memory-footprint proxy."
    ),
}

with open("results/all_paper_numbers.json", "w") as f:
    json.dump(all_paper_numbers, f, indent=2)
print("\nWrote results/all_paper_numbers.json")


## 5. Figure A -- baseline comparison bar chart (tuned F1, per dataset)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 5), dpi=300)
for ax, name in zip(axes, DATASETS):
    vals = [get_f1(m, name, tuned=True) for m in MODEL_DISPLAY_ORDER]
    colors = ["#d62728" if m == "cleids_edge" else "#1f77b4" for m in MODEL_DISPLAY_ORDER]
    ax.bar(range(len(MODEL_DISPLAY_ORDER)), vals, color=colors)
    ax.set_xticks(range(len(MODEL_DISPLAY_ORDER)))
    ax.set_xticklabels([DISPLAY_NAMES[m] for m in MODEL_DISPLAY_ORDER], rotation=60, ha="right", fontsize=8)
    ax.set_title(name, fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.grid(True, axis="y", linestyle="--", alpha=0.6)
axes[0].set_ylabel("Tuned-threshold F1")
fig.suptitle("CLEIDS-Edge vs Baselines -- Tuned F1 by Dataset (binary task)", fontsize=13)
plt.tight_layout()
plt.savefig("figures/baseline_comparison_f1.png", dpi=300)
plt.close()
print("Saved figures/baseline_comparison_f1.png")


## 6. Figure B -- accuracy-efficiency trade-off (Contribution 1's key evidence)

Legend is placed **outside** the axes deliberately -- an earlier in-plot "lower left" placement was
found, on visual inspection, to sit exactly on top of SVM's real data point (fastest model, mid-range
F1), silently hiding it. Caught by actually looking at the figure, not just checking it rendered.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), dpi=300)
for m in BASELINE_MODELS:
    f1s = [get_f1(m, name, tuned=True) for name in DATASETS]
    lats = [get_latency(m, name)["latency_ms_mean"] for name in DATASETS]
    ax.scatter(np.mean(lats), np.mean(f1s), s=90, label=DISPLAY_NAMES[m], marker="o", alpha=0.85)

cleids_f1s = [get_f1("cleids_edge", name, tuned=True) for name in DATASETS]
cleids_lat_orig = [get_latency("cleids_edge", name, "original")["latency_ms_mean"] for name in DATASETS]
cleids_lat_quant = [get_latency("cleids_edge", name, "quantized")["latency_ms_mean"] for name in DATASETS]
mean_f1 = float(np.mean(cleids_f1s))
mean_lat_orig = float(np.mean(cleids_lat_orig))
mean_lat_quant = float(np.mean(cleids_lat_quant))

ax.scatter(mean_lat_orig, mean_f1, s=220, marker="*", color="#d62728", label="CLEIDS-Edge (original)", zorder=5)
ax.scatter(mean_lat_quant, mean_f1, s=220, marker="*", color="#2ca02c", label="CLEIDS-Edge (16x8 quantized)", zorder=5)
ax.annotate("", xy=(mean_lat_quant, mean_f1), xytext=(mean_lat_orig, mean_f1),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.5))

ax.set_xscale("log")
ax.set_xlabel("Mean latency across 5 datasets (ms/sample, CPU-only, batch=1, log scale)")
ax.set_ylabel("Mean tuned-threshold F1 across 5 datasets")
ax.set_title("Accuracy-Efficiency Trade-off: CLEIDS-Edge vs Baselines")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig("figures/efficiency_frontier.png", dpi=300)
plt.close()
print("Saved figures/efficiency_frontier.png")


## 7. Figure C -- CLEIDS-Edge compression trade-off (accuracy vs size vs latency)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=300)
x = np.arange(len(DATASETS))
width = 0.35

orig_acc = [compression_results[f"cleids_edge_{n}_binary"]["original"]["accuracy"] for n in DATASETS]
quant_acc = [compression_results[f"cleids_edge_{n}_binary"]["quantized"]["accuracy"] for n in DATASETS]
axes[0].bar(x - width/2, orig_acc, width, label="Original", color="#1f77b4")
axes[0].bar(x + width/2, quant_acc, width, label="16x8 Quantized", color="#2ca02c")
axes[0].set_xticks(x); axes[0].set_xticklabels(DATASETS, rotation=30, ha="right")
axes[0].set_ylabel("Accuracy"); axes[0].set_title("Accuracy"); axes[0].set_ylim(0, 1.05)
axes[0].grid(True, axis="y", linestyle="--", alpha=0.6); axes[0].legend(fontsize=8)

orig_size = [get_latency("cleids_edge", n, "original")["size_mb"] for n in DATASETS]
quant_size = [get_latency("cleids_edge", n, "quantized")["size_mb"] for n in DATASETS]
axes[1].bar(x - width/2, orig_size, width, label="Original", color="#1f77b4")
axes[1].bar(x + width/2, quant_size, width, label="16x8 Quantized", color="#2ca02c")
axes[1].set_xticks(x); axes[1].set_xticklabels(DATASETS, rotation=30, ha="right")
axes[1].set_ylabel("Model size (MB)"); axes[1].set_title("Model Size")
axes[1].grid(True, axis="y", linestyle="--", alpha=0.6); axes[1].legend(fontsize=8)

orig_lat = [get_latency("cleids_edge", n, "original")["latency_ms_mean"] for n in DATASETS]
quant_lat = [get_latency("cleids_edge", n, "quantized")["latency_ms_mean"] for n in DATASETS]
axes[2].bar(x - width/2, orig_lat, width, label="Original", color="#1f77b4")
axes[2].bar(x + width/2, quant_lat, width, label="16x8 Quantized", color="#2ca02c")
axes[2].set_xticks(x); axes[2].set_xticklabels(DATASETS, rotation=30, ha="right")
axes[2].set_ylabel("Latency (ms/sample)"); axes[2].set_title("Latency")
axes[2].set_yscale("log")
axes[2].grid(True, axis="y", linestyle="--", alpha=0.6); axes[2].legend(fontsize=8)

fig.suptitle("CLEIDS-Edge Compression Trade-off (Binary Task): Accuracy vs Size vs Latency", fontsize=13)
plt.tight_layout()
plt.savefig("figures/compression_tradeoff.png", dpi=300)
plt.close()
print("Saved figures/compression_tradeoff.png")


## 8. Figure D -- cross-model F1 heatmap (Contribution 3's compact evidence)

Independently reconfirms two findings already documented from earlier notebooks (not new anomalies,
cross-checked against project brief SS2c and SS2e): the near-uniform ~0.465 IoT-23 column across every
model, and Standalone LSTM's real TON_IoT training collapse (F1=0.527, visibly lighter than the rest of
that row).

In [ ]:
matrix = np.array([[get_f1(m, n, tuned=True) for n in DATASETS] for m in MODEL_DISPLAY_ORDER])
fig, ax = plt.subplots(figsize=(9, 6), dpi=300)
sns.heatmap(matrix, annot=True, fmt=".3f", cmap="Blues",
            xticklabels=DATASETS, yticklabels=[DISPLAY_NAMES[m] for m in MODEL_DISPLAY_ORDER],
            ax=ax, cbar_kws={"label": "Tuned-threshold F1"})
ax.set_title("CLEIDS-Edge vs Baselines -- Tuned F1 Heatmap (binary task)")
plt.tight_layout()
plt.savefig("figures/cross_model_f1_heatmap.png", dpi=300)
plt.close()
print("Saved figures/cross_model_f1_heatmap.png")


## 9. Final summary

In [ ]:
print("=" * 90)
print("CLEIDS-Edge -- Notebook 07 Headline Summary")
print("=" * 90)
print(f"Mean tuned F1 across 5 datasets: CLEIDS-Edge={mean_f1:.4f}")
for m in BASELINE_MODELS:
    f1s = [get_f1(m, name, tuned=True) for name in DATASETS]
    print(f"  {DISPLAY_NAMES[m]:22s} mean_f1={np.mean(f1s):.4f}")
print()
print(f"CLEIDS-Edge mean latency: original={mean_lat_orig:.3f}ms, quantized={mean_lat_quant:.3f}ms "
      f"({mean_lat_orig/mean_lat_quant:.1f}x speedup, accuracy unchanged)")
print()
print("Figures written to figures/: baseline_comparison_f1.png, efficiency_frontier.png,")
print("  compression_tradeoff.png, cross_model_f1_heatmap.png")
print("Consolidated numbers written to results/all_paper_numbers.json")


## 10. Final backup + push

In [ ]:
import shutil

shutil.copy2("results/all_paper_numbers.json", os.path.join(DRIVE_RESULTS, "all_paper_numbers.json"))
for fn in os.listdir("figures"):
    if fn.startswith(("baseline_comparison", "efficiency_frontier", "compression_tradeoff", "cross_model_f1")):
        shutil.copy2(os.path.join("figures", fn), os.path.join(DRIVE_FIGURES, fn))

subprocess.run(["git", "-C", REPO_DIR, "add", "-A", "results/", "figures/", "notebooks/07_Final_Results_and_Figures.ipynb"], check=False)
commit_res = subprocess.run(
    ["git", "-C", REPO_DIR, "commit", "-m", "Notebook 07: final consolidated results + figures"],
    capture_output=True, text=True,
)
print(commit_res.stdout, commit_res.stderr)
if commit_res.returncode == 0:
    subprocess.run(["git", "-C", REPO_DIR, "push", "origin", "HEAD"], check=True)
    print("Pushed final results.")
else:
    print("Nothing new to commit (or commit failed) -- see output above.")
